## Diff vs CRPS — dataset comparison (2026-03-08)

This notebook compares AutoCast evaluation outputs for runs whose `run_name` starts with `diff_` (diffusion) versus `crps_` (CRPS).

**Datasets (initial focus)**
- Shallow water (`shallow_water2d_*`)
- Conditioned Navier–Stokes (`conditioned_navier_stokes_2d_*`)
- Lattice Boltzmann (`lattice_boltzmann_*`)

**Metrics**
- Accuracy: VRMSE (overall + rollout windows)
- Coverage: `coverage` (overall + rollout windows) + optional coverage curves
- Efficiency/complexity: training epoch time, inference latency/throughput, params, GFLOPs


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# If running from the repo root, this relative path should work.
OUTPUTS_DIR = "../../isambard-ai/outputs/2026-03-08"

DATASET_ALLOWLIST = {
    "shallow_water2d": "Shallow water",
    "conditioned_navier_stokes_2d": "Conditioned Navier–Stokes",
    "lattice_boltzmann": "Lattice Boltzmann",
}

ROLL_WINDOWS = ["0-1", "0-4", "6-12", "13-30", "31-99"]

pd.set_option("display.max_columns", 200)
plt.rcParams.update({"figure.dpi": 120})


In [ ]:
# Import RunCollator (works in the project environment where dependencies are installed)
from autocast.scripts.utils import RunCollator

params = {
    "processor_model": "model.processor._target_",
    "processor_hidden_channels": "model.processor.hidden_channels",
}

collator = RunCollator(config_params=params, outputs_dir=OUTPUTS_DIR)

df_metrics, metadata_by_run = collator.collate(
    save_csv=False,
    include_metadata_dataframes=True,
)

df_metrics.head()

In [ ]:
def normalize_dataset_family(dataset: str | None) -> str | None:
    if dataset is None or (isinstance(dataset, float) and pd.isna(dataset)):
        return None
    s = str(dataset)
    # Drop a trailing run/hash suffix like _9057134 or _5e1f575 or _f530ec5.
    m = re.match(r"^(?P<base>.+)_(?P<suffix>[0-9a-f]{6,}|\d{6,})$", s)
    base = m.group("base") if m else s

    # Normalize common dataset bases.
    if base.startswith("shallow_water2d"):
        return "shallow_water2d"
    if base.startswith("conditioned_navier_stokes_2d"):
        return "conditioned_navier_stokes_2d"
    if base.startswith("lattice_boltzmann"):
        return "lattice_boltzmann"
    return base


def loss_family_from_run_name(run_name: str | None) -> str | None:
    if not run_name:
        return None
    m = re.match(r"^(diff|crps)_", str(run_name))
    return m.group(1) if m else None


df = df_metrics.copy()

df["loss_family"] = df["run_name"].map(loss_family_from_run_name)

df["dataset_raw"] = df.get("dataset")

df["dataset_family"] = df["dataset_raw"].map(normalize_dataset_family)

df["dataset_label"] = df["dataset_family"].map(DATASET_ALLOWLIST)

# Filter to the three datasets.
df = df[df["dataset_family"].isin(DATASET_ALLOWLIST.keys())].copy()

# Keep only diff/crps runs.
df = df[df["loss_family"].isin(["diff", "crps"])].copy()

# Basic sanity view.
_cols = ["run_name", "loss_family", "dataset_raw", "dataset_family", "dataset_label"]
_cols = [c for c in _cols if c in df.columns]
_view = df[_cols].copy() if _cols else df.copy()
_sort = [c for c in ["dataset_family", "loss_family", "run_name"] if c in _view.columns]
if _sort:
    _view = _view.sort_values(_sort)
_view.head(30)

In [ ]:
# Flatten timing/complexity/benchmark metadata into wide columns and merge onto df.
from autocast.scripts.utils import flatten_metadata_by_run

md_wide = flatten_metadata_by_run(metadata_by_run)

# Merge onto the filtered df (left join: keep only the runs we filtered to).
df = df.merge(md_wide, on="run_path", how="left")

# Inspect what we got
cols_meta = sorted(
    [c for c in df.columns if any(k in c for k in ["train_", "params_", "model_", "rollout_"])]
)
cols_meta[:60], df.shape

## Tables

The cells below build:
- a **per-run table**
- a **grouped summary** (mean/std) by `(dataset, loss_family)`
- **diff − crps deltas** per dataset


In [ ]:
def existing_cols(cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]

metric_cols = [
    "overall_vrmse",
    "overall_coverage",
] + [f"vrmse_{w}" for w in ROLL_WINDOWS] + [f"coverage_{w}" for w in ROLL_WINDOWS]

perf_cols = existing_cols(metric_cols)

speed_cols = existing_cols(
    [
        "train_mean_epoch_s",
        "train_total_s",
        "params_model_total",
        "params_model_trainable",
        "model_latency_ms_per_sample",
        "model_latency_ms_per_batch",
        "model_throughput_samples_per_sec",
        "model_gflops_per_sample",
        "model_peak_gpu_memory_mb",
        "rollout_latency_ms_per_step",
        "rollout_latency_ms_per_sample",
        "rollout_throughput_samples_per_sec",
        "rollout_peak_gpu_memory_mb",
    ]
)

id_cols = existing_cols(
    [
        "run_name",
        "run_path",
        "dataset_label",
        "dataset_family",
        "loss_family",
        "processor_model",
        "processor_hidden_channels",
    ]
)

per_run = df[id_cols + perf_cols + speed_cols].copy()

sort_cols = [c for c in ["dataset_family", "loss_family", "run_name"] if c in per_run.columns]
if sort_cols:
    per_run = per_run.sort_values(sort_cols)

per_run

In [ ]:
group_keys = ["dataset_family", "dataset_label", "loss_family"]

summary_cols = existing_cols(
    [
        "overall_vrmse",
        "overall_coverage",
        "train_mean_epoch_s",
        "model_latency_ms_per_sample",
        "model_throughput_samples_per_sec",
        "rollout_latency_ms_per_step",
        "rollout_throughput_samples_per_sec",
        "params_model_total",
        "model_gflops_per_sample",
    ]
    + [f"vrmse_{w}" for w in ROLL_WINDOWS]
    + [f"coverage_{w}" for w in ROLL_WINDOWS]
)

summary = (
    df.groupby(group_keys)[summary_cols]
    .agg(["mean", "std", "count"])
    .sort_index()
)
summary

In [ ]:
# Diff - CRPS deltas (per dataset) for primary metrics.
primary = existing_cols(
    [
        "overall_vrmse",
        "overall_coverage",
        "train_mean_epoch_s",
        "model_latency_ms_per_sample",
        "model_throughput_samples_per_sec",
        "params_model_total",
        "model_gflops_per_sample",
    ]
)

means = df.groupby(["dataset_family", "loss_family"])[primary].mean(numeric_only=True)

# Wide: columns like ('overall_vrmse','diff')
wide = means.unstack("loss_family")

col_families = set(wide.columns.get_level_values(1)) if isinstance(wide.columns, pd.MultiIndex) else set()

if {"diff", "crps"}.issubset(col_families):
    try:
        deltas = wide.xs("diff", axis=1, level=1) - wide.xs("crps", axis=1, level=1)
        deltas = deltas.rename(columns={c: f"diff_minus_crps__{c}" for c in deltas.columns})
    except KeyError as e:
        print(f"Could not compute diff-crps deltas: {e}")
        deltas = pd.DataFrame(index=wide.index)
else:
    print(f"Missing diff/crps columns in wide means: {sorted(col_families)}")
    deltas = pd.DataFrame(index=wide.index)

# Attach friendly dataset labels.
deltas = deltas.reset_index().merge(
    df[["dataset_family", "dataset_label"]].drop_duplicates(), on="dataset_family", how="left"
)
deltas = deltas.set_index(["dataset_family", "dataset_label"]).sort_index()

deltas

## Plots

All plots below compare **diff vs crps** within each dataset.


In [ ]:
def grouped_bar(df_in: pd.DataFrame, metric: str, title: str, ylabel: str):
    required = ["dataset_label", "loss_family", metric]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        print(f"Skipping {metric}: missing columns {missing}")
        return

    d = df_in[required].dropna().copy()
    if d.empty:
        print(f"No data for {metric}")
        return

    # Aggregate to mean per group.
    g = d.groupby(["dataset_label", "loss_family"], dropna=False)[metric].mean().reset_index()

    datasets = list(g["dataset_label"].drop_duplicates())
    families = [f for f in ["crps", "diff"] if f in set(g["loss_family"]) ]

    x = range(len(datasets))
    width = 0.38

    fig, ax = plt.subplots(figsize=(8, 3.6))
    for i, fam in enumerate(families):
        vals = (
            g[g["loss_family"] == fam]
            .set_index("dataset_label")
            .reindex(datasets)[metric]
            .values
        )
        offs = (
            (-width / 2)
            if len(families) == 2 and i == 0
            else (width / 2 if len(families) == 2 else 0)
        )
        ax.bar([xi + offs for xi in x], vals, width=width, label=fam)

    ax.set_xticks(list(x))
    ax.set_xticklabels(datasets, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    plt.show()


grouped_bar(df, "overall_vrmse", "Overall VRMSE (lower is better)", "VRMSE")


In [ ]:
if "overall_coverage" in df.columns:
    grouped_bar(df, "overall_coverage", "Overall coverage (higher is better)", "Coverage")

# Optional: rollout windows
for w in ROLL_WINDOWS:
    col = f"vrmse_{w}"
    if col in df.columns:
        grouped_bar(df, col, f"Rollout VRMSE window {w}", "VRMSE")
        break  # keep notebook compact by default


In [ ]:
# Coverage vs VRMSE scatter (points = runs)
if {"overall_vrmse", "overall_coverage"}.issubset(df.columns):
    _cols = ["dataset_label", "loss_family", "overall_vrmse", "overall_coverage", "run_name"]
    _cols = [c for c in _cols if c in df.columns]
    d = df[_cols].dropna()

    fig, axes = plt.subplots(1, len(DATASET_ALLOWLIST), figsize=(12, 3.3), sharex=False, sharey=False)
    if len(DATASET_ALLOWLIST) == 1:
        axes = [axes]

    for ax, (ds_key, ds_label) in zip(axes, DATASET_ALLOWLIST.items()):
        sub = d[d.get("dataset_label") == ds_label] if "dataset_label" in d.columns else d
        for fam, marker in [("crps", "o"), ("diff", "^")]:
            ss = sub[sub.get("loss_family") == fam] if "loss_family" in sub.columns else sub
            if ss.empty:
                continue
            ax.scatter(ss["overall_vrmse"], ss["overall_coverage"], label=fam, marker=marker, alpha=0.85)
        ax.set_title(ds_label)
        ax.set_xlabel("overall_vrmse")
        ax.set_ylabel("overall_coverage")
        ax.grid(alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=2)
    plt.tight_layout(rect=(0, 0, 1, 0.9))
    plt.show()
else:
    print("Missing overall_vrmse/overall_coverage")


In [ ]:
# Timing / throughput / complexity comparisons (group means)
for metric, title, ylabel in [
    ("train_mean_epoch_s", "Mean epoch time (s) (lower is better)", "seconds"),
    ("model_latency_ms_per_sample", "Model latency per sample (ms) (lower is better)", "ms"),
    ("model_throughput_samples_per_sec", "Model throughput (samples/s) (higher is better)", "samples/s"),
    ("params_model_total", "Model parameters (total)", "params"),
    ("model_gflops_per_sample", "GFLOPs per sample", "GFLOPs"),
]:
    if metric in df.columns:
        grouped_bar(df, metric, title, ylabel)


## Optional: coverage curves (calibration)

These plots read the per-run curve CSVs (`test_coverage_window_all.csv` and `rollout_coverage_window_*.csv`).


In [ ]:
def load_curve_csv(run_path: str, filename: str) -> pd.DataFrame | None:
    p = Path(OUTPUTS_DIR).expanduser().resolve() / run_path / "eval" / filename
    if not p.exists():
        return None
    c = pd.read_csv(p)
    c = c.copy()
    c["run_path"] = run_path
    return c


def plot_coverage_curves(df_in: pd.DataFrame, curve_filename: str, title: str, max_runs_per_group: int = 6):
    # Sample a few runs per dataset/family to keep plots readable.
    d = df_in[["run_path", "dataset_label", "loss_family"]].drop_duplicates().copy()

    sampled = (
        d.groupby(["dataset_label", "loss_family"], dropna=False)
        .head(max_runs_per_group)
        .reset_index(drop=True)
    )

    curves = []
    for _, r in sampled.iterrows():
        c = load_curve_csv(r["run_path"], curve_filename)
        if c is None or c.empty:
            continue
        c["dataset_label"] = r["dataset_label"]
        c["loss_family"] = r["loss_family"]
        curves.append(c)

    if not curves:
        print(f"No curves found for {curve_filename}")
        return

    cur = pd.concat(curves, ignore_index=True)

    fig, axes = plt.subplots(1, len(DATASET_ALLOWLIST), figsize=(12, 3.3), sharex=True, sharey=True)
    if len(DATASET_ALLOWLIST) == 1:
        axes = [axes]

    # Consistent style per family + figure-level legend.
    family_style = {
        "crps": {"color": "tab:blue", "label": "CRPS"},
        "diff": {"color": "tab:orange", "label": "Diffusion"},
    }

    for ax, (ds_key, ds_label) in zip(axes, DATASET_ALLOWLIST.items()):
        sub = cur[cur["dataset_label"] == ds_label]
        if sub.empty:
            ax.set_title(ds_label)
            continue

        # Perfect calibration line.
        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, alpha=0.6, color="black")

        for fam, alpha in [("crps", 0.6), ("diff", 0.6)]:
            ss = sub[sub["loss_family"] == fam]
            if ss.empty:
                continue
            style = family_style.get(fam, {"color": None, "label": fam})
            for run_path, sss in ss.groupby("run_path"):
                if {"coverage_level", "observed_mean"}.issubset(sss.columns):
                    ax.plot(
                        sss["coverage_level"],
                        sss["observed_mean"],
                        alpha=alpha,
                        color=style["color"],
                    )

        ax.set_title(ds_label)
        ax.set_xlabel("coverage_level")
        ax.set_ylabel("observed_mean")
        ax.grid(alpha=0.25)

    # Family-level legend at figure scope.
    legend_handles = []
    for fam in ["crps", "diff"]:
        if fam in set(cur["loss_family"].dropna().unique()):
            style = family_style.get(fam, {"color": None, "label": fam})
            handle = plt.Line2D([0], [0], color=style["color"], linewidth=2, label=style["label"])
            legend_handles.append(handle)
    legend_handles.append(
        plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1, label="Perfect calibration")
    )
    fig.legend(handles=legend_handles, loc="upper center", ncol=max(2, len(legend_handles)))

    plt.suptitle(title)
    plt.tight_layout(rect=(0, 0, 1, 0.9))
    plt.show()


plot_coverage_curves(
    df,
    curve_filename="test_coverage_window_all.csv",
    title="Test coverage calibration curves (window=all)",
)

In [ ]:
# Example rollout window curve (adjust filename as needed)
plot_coverage_curves(
    df,
    # curve_filename="rollout_coverage_window_0-4.csv",
    curve_filename="rollout_coverage_window_6-12.csv",
    title="Rollout coverage calibration curves (window 0-4)",
)